# Project 29: Root Cause Analysis Statistics
Notebook ini mencatat setiap uji, syarat, hasil, dan batas kesimpulannya. Data simulasi; angka menguji metode, bukan kondisi pabrik. Aturan: `project/docs/analysis_rules.md`.

In [1]:
import duckdb, pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
pd.options.display.float_format = '{:.4f}'.format
R = r'/home/claude/project-29-root-cause-analysis-statistics/project/results/2-analysis'
con = duckdb.connect(r'/home/claude/project-29-root-cause-analysis-statistics/project/rca.duckdb', read_only=True)
tr = pd.read_csv(R + '/test_results.csv')

## Q1. Pareto defect
Jenis defect yang menyumbang sekitar 80% NG.

In [2]:
pd.read_csv(R + '/pareto.csv')

,defect_code,ng,share,cum_share
0,LED-OPEN,398,0.3097,0.3097
1,SOLDER,311,0.2420,0.5518
2,DRIVER,250,0.1946,0.7463
3,HOUSING,190,0.1479,0.8942
4,COSMETIC,136,0.1058,1.0000


## Q2. NG per stasiun inspeksi
Tidak bisa dijawab: data simulasi v1 tidak punya kolom `inspect_station` (lihat Report bagian 3).

In [3]:
con.execute('DESCRIBE unit_result').df()[['column_name']].T

,0,1,2,3,4,5,6,7
column_name,unit_id,inspect_date,model,shift,operator_id,lot_id,result,defect_code


## Q3. Shift 1 vs Shift 2 (chi-square Pearson, tanpa koreksi kontinuitas)

In [4]:
t = con.execute("SELECT shift, sum(ng) ng, count(*)-sum(ng) ok FROM v_unit GROUP BY shift ORDER BY shift").df()
print(t)
chi2, p, dof, exp = chi2_contingency(t[['ng','ok']].values, correction=False)
print('chi2 =', round(chi2, 3), 'p =', round(p, 4), 'expected minimal =', round(exp.min(), 1))

     shift       ng         ok
0  Shift 1 598.0000 24342.0000
1  Shift 2 687.0000 24373.0000
chi2 = 5.896 p = 0.0152 expected minimal = 641.0


## Q4 dan Q5. Operator dan lot: uji per level dengan koreksi Benjamini-Hochberg
p mentah bisa tampak signifikan; setelah koreksi karena banyak perbandingan hanya yang kuat yang bertahan.

In [5]:
o = tr[tr.factor=='operator_id'][['level','n','ng','rate','test','p_value','p_adj','p_adj_bonf','odds_ratio','status']]
o.sort_values('p_value').head(6)

,level,n,ng,rate,test,p_value,p_adj,p_adj_bonf,odds_ratio,status
20,OP19,2887,178,0.0617,Chi-square,0.0000,0.0000,0.0000,2.7273,Terbukti
9,OP08,464,20,0.0431,Chi-square,0.0177,0.1766,0.3531,1.7161,Noise
7,OP06,3043,60,0.0197,Chi-square,0.0303,0.2022,0.6067,0.7495,Noise
21,OP20,39,3,0.0769,Fisher exact,0.0785,0.3846,1.0000,3.1591,Noise
16,OP15,2435,50,0.0205,Chi-square,0.0961,0.3846,1.0000,0.7851,Noise
11,OP10,2427,52,0.0214,Chi-square,0.1687,0.4002,1.0000,0.8214,Noise


In [6]:
l = tr[tr.factor=='lot_id'][['level','n','ng','rate','test','p_value','p_adj','odds_ratio','status']]
l.sort_values('p_value').head(6)

,level,n,ng,rate,test,p_value,p_adj,odds_ratio,status
45,LB29,1542,54,0.0350,Chi-square,0.0188,0.6990,1.3924,Noise
39,LB11,1136,40,0.0352,Chi-square,0.0404,0.6990,1.3960,Noise
58,LC30,1122,39,0.0348,Chi-square,0.0524,0.6990,1.3767,Noise
35,LA40,1118,20,0.0179,Chi-square,0.0950,0.7839,0.6856,Noise
54,LC18,1149,21,0.0183,Chi-square,0.1076,0.7839,0.7009,Noise
60,LC36,1132,37,0.0327,Chi-square,0.1330,0.7839,1.2894,Noise


In [7]:
tr[tr.factor=='supplier'][['level','n','ng','rate','p_value','p_adj','odds_ratio','status']]

,level,n,ng,rate,p_value,p_adj,odds_ratio,status
62,Supplier A,17623,424,0.0241,0.0866,0.1299,0.9021,Noise
63,Supplier B,16054,443,0.0276,0.0653,0.1299,1.1160,Noise
64,Supplier C,16011,410,0.0256,0.9280,0.9280,0.9945,Noise


## Q6 dan Q8. Model lampu, uji keseluruhan, dan Cramér's V

In [8]:
pd.read_csv(R + '/factor_tests.csv')

,factor,levels,n,chi2,dof,p_value,cramers_v,cells_exp_lt5,cells,expected_min
0,shift,2,50000,5.8960,1,0.0152,0.0109,0,4,640.9580
1,operator_id,20,49687,178.3489,19,0.0000,0.0599,1,40,1.0039
2,lot_id,40,49688,38.4910,39,0.4929,0.0278,0,80,28.2447
3,supplier,3,49688,4.2005,2,0.1224,0.0092,0,6,411.4886
4,model,3,50000,7.3834,2,0.0249,0.0122,0,6,319.7594
5,experience,2,49687,161.7260,1,0.0000,0.0571,0,4,75.3186


## Q7. Odds ratio dan CI 95% untuk level yang terbukti

In [9]:
tr[tr.status=='Terbukti'][['factor','level','n','rate','odds_ratio','or_ci_low','or_ci_high','p_adj']]

,factor,level,n,rate,odds_ratio,or_ci_low,or_ci_high,p_adj
1,shift,Shift 2,25060,0.0274,1.1474,1.0268,1.2821,0.0152
20,operator_id,OP19,2887,0.0617,2.7273,2.3170,3.2102,0.0000
65,model,12W,12442,0.0289,1.1800,1.0430,1.3351,0.0256
68,experience,baru,2926,0.0619,2.7422,2.3323,3.2241,0.0000


## Q9 dan Q10. Regresi logistik dan faktor yang saling terkait

In [10]:
pd.read_csv(R + '/regression.csv')

,model,term,odds_ratio,or_low,or_high,p_value,p_adj
0,Terpisah: shift,Shift 2,1.1460,1.0249,1.2814,0.0167,0.0167
1,Terpisah: model,Model 5W,0.9511,0.8336,1.0851,0.4560,0.4560
2,Terpisah: model,Model 12W,1.1617,1.0121,1.3333,0.0331,0.0662
3,Terpisah: supplier,Supplier B,1.1427,0.9983,1.3081,0.0530,0.1060
4,Terpisah: supplier,Supplier C,1.0586,0.9224,1.2149,0.4179,0.4179
5,Terpisah: pengalaman operator,Operator baru,2.7432,2.3322,3.2267,0.0000,0.0000
6,Bersamaan: shift + model + supplier + pengalaman,Shift 2,0.9556,0.8476,1.0774,0.4584,0.4584
7,Bersamaan: shift + model + supplier + pengalaman,Model 5W,0.9460,0.8290,1.0796,0.4105,0.4584
8,Bersamaan: shift + model + supplier + pengalaman,Model 12W,1.1624,1.0125,1.3346,0.0326,0.0925
9,Bersamaan: shift + model + supplier + pengalaman,Supplier B,1.1476,1.0023,1.3140,0.0463,0.0925


In [11]:
pd.read_csv(R + '/confounding.csv')

,comparison,rate_shift2,rate_shift1,n_shift2,n_shift1,p_value,odds_ratio,or_low,or_high
0,"Shift 2 vs Shift 1, semua unit",0.0274,0.0240,25060,24940,0.0152,1.1474,1.0268,1.2821
1,Shift 2 tanpa OP19 vs Shift 1,0.0230,0.0240,22173,24940,0.4649,0.9564,0.8486,1.0779


In [12]:
pd.read_csv(R + '/lr_tests.csv')

,factor,lr_chi2,dof,p_value
0,Operator,137.2469,19,0.0000
1,Lot chip LED,39.1019,39,0.4653
2,Model lampu,8.1517,2,0.0170


## Q11. Dampak (estimated, simulated)

In [13]:
pd.read_csv(R + '/impact.csv')

,factor,level,n,ng,rate,baseline_rate,ng_avoided_period,ng_avoided_low,ng_avoided_high,ng_avoided_year,share_of_all_ng,label
0,operator_id,OP19,2887,178,0.0617,0.0235,110.0815,86.3939,137.1323,220.1629,0.0861,"estimated, simulated"
1,model,12W,12442,360,0.0289,0.0246,53.5713,18.6956,92.0650,107.1426,0.0417,"estimated, simulated"
2,experience,baru,2926,181,0.0619,0.0235,112.2943,88.3966,139.5537,224.5886,0.0878,"estimated, simulated"


## Q12. Validasi terhadap ground truth

In [14]:
pd.read_csv(R + '/ground_truth_check.csv').style.hide(axis='index')

fact_no,fakta,ditemukan,hasil
1,"Dua lot Supplier B punya defect rate ±1,6x rata-rata","LB11 dan LB29 adalah satu-satunya lot dengan p-value mentah di bawah 0.05 (p 0.040 dan 0.019), tetapi tidak ada lot yang lolos koreksi (p BH terkecil 0.70); uji rasio kemungkinan semua lot p 0.47; Supplier B p BH 0.13",Sebagian
2,Efek shift sebenarnya berasal dari operator baru (hanya Shift 2),"Shift 2 tampak lebih tinggi (p 0.015); tanpa OP19 selisih hilang (p 0.46) dan setelah dikontrol pengalaman operator odds ratio shift 0.96 (p 0.46); OP19 odds ratio 2.7, p BH < 0.001",Sesuai
3,"Operator noise: sedikit lebih tinggi secara acak, tidak boleh signifikan setelah koreksi","OP08 defect rate 4.31%: p mentah 0.018 (tampak signifikan), p BH 0.18 (tidak signifikan setelah koreksi); 3 dari 20 operator signifikan tanpa koreksi, 1 setelah koreksi",Sesuai
4,"Beberapa operator hanya mengerjakan kurang dari 100 unit, sehingga butuh Fisher exact","Hanya satu operator (OP20, 39 unit, expected count minimal 1.0) yang memakai Fisher exact; p 0.079. Operator lain punya paling sedikit 464 unit",Sebagian
5,Sekitar 2-3% data kotor,1.250 baris bermasalah (2.5%); seluruhnya ditemukan dan sama persis dengan unit_id yang ditanam (R1 sampai R4),Sesuai


## Q13. Syarat uji dan data tidak valid

In [15]:
pd.read_csv(R.replace('2-analysis','1-data-preparation') + '/cleaning_log.csv')

,rule,problem,rows_found,rows_planted,rows_dropped,rows_fixed,decision,match_ground_truth
0,R1,Baris ganda (unit_id sama),312,312,312,0,"Baris kedua dibuang, satu baris per unit diper...",ya
1,R2,"Hasil tidak baku (ok, PASS, ng, FAIL, spasi)",313,313,0,313,Dibakukan menjadi OK atau NG,ya
2,R3,operator_id kosong,313,313,0,0,Baris dipertahankan; dikeluarkan hanya dari uj...,ya
3,R4,lot_id tidak dikenal (LX99),312,312,0,0,Baris dipertahankan; lot dan supplier dikosong...,ya
4,R5,Pemeriksaan konsistensi (6 aturan),0,0,0,0,Tidak ada pelanggaran; data tidak diubah,ya


In [16]:
pd.read_csv(R + '/sensitivity.csv').T

,0,1,2
treatment,A. Data mentah apa adanya,B. Semua unit bermasalah dibuang,C. Cleaning sesuai log (dipakai)
n,50312,48750,50000
ng_rate,0.0255,0.0257,0.0257
shift_p,0.0162,0.0079,0.0152
shift_rate1,0.0238,0.0238,0.0240
shift_rate2,0.0272,0.0276,0.0274
operators_tested,20,20,20
operators_sig_raw,3,3,3
operators_sig_bh,1,1,1
op19_or,2.7257,2.7405,2.7273


## Q14. Jumlah unit yang dibutuhkan (power 80%)

In [17]:
pd.read_csv(R + '/power.csv')

,scope,rate_multiplier,alpha_rule,alpha_used,n_needed_per_level,n_median_actual,power_at_actual_n
0,Lot chip LED (40 uji),1.3000,"tanpa koreksi (0,05)",0.0500,3867.8137,1152.5000,0.3336
1,Lot chip LED (40 uji),1.3000,"Bonferroni (0,05 / jumlah uji)",0.0013,8158.3078,1152.5000,0.0448
2,Lot chip LED (40 uji),1.6000,"tanpa koreksi (0,05)",0.0500,1078.8787,1152.5000,0.8253
3,Lot chip LED (40 uji),1.6000,"Bonferroni (0,05 / jumlah uji)",0.0013,2275.6590,1152.5000,0.3701
4,Lot chip LED (40 uji),2.0000,"tanpa koreksi (0,05)",0.0500,439.1298,1152.5000,0.9950
5,Lot chip LED (40 uji),2.0000,"Bonferroni (0,05 / jumlah uji)",0.0013,926.2483,1152.5000,0.9051
6,Lot chip LED (40 uji),2.6000,"tanpa koreksi (0,05)",0.0500,199.4203,1152.5000,1.0000
7,Lot chip LED (40 uji),2.6000,"Bonferroni (0,05 / jumlah uji)",0.0013,420.6335,1152.5000,0.9998
8,Operator (20 uji),1.3000,"tanpa koreksi (0,05)",0.0500,3969.5983,2505.5000,0.6048
9,Operator (20 uji),1.3000,"Bonferroni (0,05 / jumlah uji)",0.0025,7554.9319,2505.5000,0.2126


## Cek silang
SQL vs pandas dan SQL/Python vs Excel (dihitung LibreOffice).

In [18]:
pd.concat([pd.read_csv(R + '/crosscheck.csv').status.value_counts(), pd.read_csv(R + '/crosscheck_excel.csv').status.value_counts()], axis=1, keys=['pandas','Excel'])

,pandas,Excel
status,,
PASS,75,65


## Batas kesimpulan
Uji statistik menunjukkan keterkaitan, bukan sebab. Lot yang ditanam tidak terbukti pada ukuran data ini (Q14 menjelaskan alasannya). Hasil per operator dipakai untuk pelatihan, bukan menilai orang.